<a href="https://colab.research.google.com/github/qasim5570/databricks-apache-spark-developer-2026/blob/main/Fraud_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""CIS Fraud Detection — with Validation, Hyperparameter Tuning, and Final Test Evaluation

Workflow:
  1. Fit preprocessing pipeline on X_train only
  2. Transform train, val, and test using the same fitted pipeline
  3. Use GridSearchCV on train+val to find best model hyperparameters
  4. Refit best model on ALL labelled data (train + val)
  5. Evaluate on test set — exactly once
"""

# ── Imports ──────────────────────────────────────────────────────────────────
import os
import gc
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)

from xgboost import XGBClassifier
import shap

from google.colab import drive
drive.mount('/content/drive')
os.chdir("/content/drive/MyDrive/fraud_detection")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — HELPERS & CUSTOM TRANSFORMERS  (unchanged from v1)
# ─────────────────────────────────────────────────────────────────────────────

def get_missing_report(df, threshold=0):
    null_counts = df.isnull().sum()
    null_perc   = (null_counts / len(df)) * 100
    report = pd.DataFrame({"count": null_counts, "percent": null_perc})
    return report[report["count"] > threshold].sort_values("percent", ascending=False)


class DropHighMissingCols(BaseEstimator, TransformerMixin):
    def __init__(self, limit=70.0):
        self.limit = limit
        self.cols_to_drop_ = []

    def fit(self, X, y=None):
        report = get_missing_report(X)
        self.cols_to_drop_ = report[report["percent"] > self.limit].index.tolist()
        print(f"[DropHighMissingCols] Dropping {len(self.cols_to_drop_)} columns > {self.limit}% missing.")
        return self

    def transform(self, X):
        return X.drop(columns=[c for c in self.cols_to_drop_ if c in X.columns])


class VColumnPCATransformer(BaseEstimator, TransformerMixin):
    def __init__(self, variance_threshold=0.90):
        self.variance_threshold = variance_threshold
        self.v_cols_       = []
        self.imputer_      = SimpleImputer(strategy="median")
        self.scaler_       = StandardScaler()
        self.pca_          = None
        self.n_components_ = 0

    def fit(self, X, y=None):
        self.v_cols_ = [c for c in X.columns if c.startswith("V")]
        v_data    = X[self.v_cols_].astype("float32")
        v_imputed = self.imputer_.fit_transform(v_data)
        v_scaled  = self.scaler_.fit_transform(v_imputed)

        pca_full = PCA().fit(v_scaled)
        cumvar   = np.cumsum(pca_full.explained_variance_ratio_)
        self.n_components_ = int(np.argmax(cumvar >= self.variance_threshold) + 1)

        self.pca_ = PCA(n_components=self.n_components_)
        self.pca_.fit(v_scaled)

        print(f"[VColumnPCATransformer] {len(self.v_cols_)} V-cols → {self.n_components_} PCA components.")
        del v_imputed, v_scaled
        gc.collect()
        return self

    def transform(self, X):
        X = X.copy()
        v_data    = X[self.v_cols_].astype("float32")
        v_imputed = self.imputer_.transform(v_data)
        v_scaled  = self.scaler_.transform(v_imputed)
        v_pca     = self.pca_.transform(v_scaled)

        pca_cols = [f"v_pca_{i}" for i in range(self.n_components_)]
        pca_df   = pd.DataFrame(v_pca, columns=pca_cols, index=X.index)

        X = X.drop(columns=self.v_cols_)
        X = pd.concat([X, pca_df], axis=1)
        del v_imputed, v_scaled, v_pca
        gc.collect()
        return X


class NumericalImputer(BaseEstimator, TransformerMixin):
    PROTECTED = {"TransactionID", "isFraud", "TransactionDT", "TransactionAmt"}

    def __init__(self, fill_value=-999):
        self.fill_value = fill_value
        self.cols_with_nan_ = []

    def fit(self, X, y=None):
        num_cols = X.select_dtypes(include=["number"]).columns
        eligible = [c for c in num_cols
                    if c not in self.PROTECTED and not c.startswith("v")]
        self.cols_with_nan_ = [c for c in eligible if X[c].isnull().any()]
        print(f"[NumericalImputer] {len(self.cols_with_nan_)} columns get _is_nan flags.")
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cols_with_nan_:
            if col in X.columns:
                X[f"{col}_is_nan"] = X[col].isnull().astype(int)
                X[col] = X[col].fillna(self.fill_value)
        return X


class TransactionDTConverter(BaseEstimator, TransformerMixin):
    def __init__(self, start_date="2025-01-01"):
        self.start_date = start_date
        self.dt_min_    = None

    def fit(self, X, y=None):
        self.dt_min_ = X["TransactionDT"].min()
        return self

    def transform(self, X):
        X = X.copy()
        origin   = pd.to_datetime(self.start_date)
        X["date"] = origin + pd.to_timedelta(X["TransactionDT"] - self.dt_min_, unit="s")
        return X


class CategoricalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cardinality_threshold=5):
        self.cardinality_threshold = cardinality_threshold
        self.cat_cols_       = []
        self.low_card_cols_  = []
        self.high_card_cols_ = []
        self.label_encoders_ = {}
        self.freq_maps_      = {}

    def fit(self, X, y=None):
        self.cat_cols_ = X.select_dtypes(include=["object"]).columns.tolist()
        X_filled       = X[self.cat_cols_].fillna("Unknown")
        unique_counts  = X_filled.nunique()

        self.low_card_cols_  = unique_counts[unique_counts <= self.cardinality_threshold].index.tolist()
        self.high_card_cols_ = unique_counts[unique_counts >  self.cardinality_threshold].index.tolist()

        for col in self.low_card_cols_:
            le = LabelEncoder()
            le.fit(X_filled[col].astype(str))
            self.label_encoders_[col] = le

        for col in self.high_card_cols_:
            self.freq_maps_[col] = X_filled[col].value_counts().to_dict()

        return self

    def transform(self, X):
        X = X.copy()
        X[self.cat_cols_] = X[self.cat_cols_].fillna("Unknown")

        for col in self.low_card_cols_:
            le    = self.label_encoders_[col]
            known = set(le.classes_)
            X[col] = X[col].astype(str).apply(
                lambda v: le.transform([v])[0] if v in known else -1
            )

        for col in self.high_card_cols_:
            X[col] = X[col].map(self.freq_maps_[col]).fillna(0).astype(int)

        return X


def build_preprocessing_pipeline():
    return Pipeline(steps=[
        ("drop_high_missing", DropHighMissingCols(limit=70.0)),
        ("v_pca",             VColumnPCATransformer(variance_threshold=0.90)),
        ("num_impute",        NumericalImputer(fill_value=-999)),
        ("dt_convert",        TransactionDTConverter(start_date="2025-01-01")),
        ("cat_encode",        CategoricalEncoder(cardinality_threshold=5)),
    ])


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — LOAD AND SPLIT DATA
# ─────────────────────────────────────────────────────────────────────────────

df_raw = pd.read_csv("train_transaction.csv")

X_raw = df_raw.drop(columns=["isFraud"])
y_raw = df_raw["isFraud"]

# ── Split into train (64%), val (16%), test (20%) ────────────────────────────
# stratify=y ensures fraud % is preserved in every split — critical for
# imbalanced datasets like fraud where only ~3.5% of rows are fraud.
X_temp, X_test_raw, y_temp, y_test = train_test_split(
    X_raw, y_raw, test_size=0.20, random_state=42, stratify=y_raw
)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.20, random_state=42, stratify=y_temp
)

print(f"Train size : {X_train_raw.shape[0]:,} rows ({y_train.mean()*100:.1f}% fraud)")
print(f"Val size   : {X_val_raw.shape[0]:,} rows ({y_val.mean()*100:.1f}% fraud)")
print(f"Test size  : {X_test_raw.shape[0]:,} rows ({y_test.mean()*100:.1f}% fraud)")

# NOTE: If you have a separate test_transaction.csv with labels, replace the
# split above with:
#   df_test = pd.read_csv("test_transaction.csv")
#   X_test_raw, y_test = df_test.drop("isFraud", axis=1), df_test["isFraud"]
# And only do one split for train/val from train_transaction.csv.


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — FIT PIPELINE ON TRAINING DATA, TRANSFORM ALL SPLITS
# ─────────────────────────────────────────────────────────────────────────────

pipeline = build_preprocessing_pipeline()
pipeline.fit(X_train_raw)                                  # ← learns from train only
joblib.dump(pipeline, "fraud_preprocessing_pipeline.pkl")

X_train_clean = pipeline.transform(X_train_raw)
X_val_clean   = pipeline.transform(X_val_raw)    # uses training parameters
X_test_clean  = pipeline.transform(X_test_raw)   # uses training parameters

# Drop date column — not a model feature
X_train_clean = X_train_clean.drop(columns=["date"], errors="ignore")
X_val_clean   = X_val_clean.drop(columns=["date"],   errors="ignore")
X_test_clean  = X_test_clean.drop(columns=["date"],  errors="ignore")

print(f"\nCleaned shapes → Train: {X_train_clean.shape}, Val: {X_val_clean.shape}, Test: {X_test_clean.shape}")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — BASELINE: QUICK COMPARISON BEFORE TUNING
# ─────────────────────────────────────────────────────────────────────────────
# Run default models on val first — tells you which algorithm is worth tuning.

rf_base  = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
xgb_base = XGBClassifier(n_estimators=100, random_state=42,
                          use_label_encoder=False, eval_metric="logloss")

rf_base.fit(X_train_clean, y_train)
xgb_base.fit(X_train_clean, y_train)

rf_val_auc  = roc_auc_score(y_val, rf_base.predict_proba(X_val_clean)[:, 1])
xgb_val_auc = roc_auc_score(y_val, xgb_base.predict_proba(X_val_clean)[:, 1])

print(f"\n── Baseline val AUC ──")
print(f"Random Forest : {rf_val_auc:.4f}")
print(f"XGBoost       : {xgb_val_auc:.4f}")
print(f"Winner        : {'XGBoost' if xgb_val_auc > rf_val_auc else 'Random Forest'}")
# → Whichever wins here is the model you tune in Section 5.


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 — HYPERPARAMETER TUNING WITH GridSearchCV
# ─────────────────────────────────────────────────────────────────────────────
# GridSearchCV systematically tries every combination of parameters you give
# it. For each combination it does k-fold cross-validation on the TRAINING
# set — it never touches X_val_clean or X_test_clean.
#
# After it finishes, you read best_params_ and best_score_ to make your
# decision. That's your answer to "which parameters should I use?"

# ── Combine train + val for cross-validation ─────────────────────────────────
# GridSearchCV does its own internal validation splits, so we give it all
# labelled data we have (train + val). It will split internally.
X_trainval_clean = pd.concat([X_train_clean, X_val_clean], axis=0)
y_trainval       = pd.concat([y_train, y_val], axis=0)

# ── Define the parameter grid ─────────────────────────────────────────────────
# Each key is a model parameter. Each value is a list of options to try.
# GridSearchCV tries every combination: 3 × 3 × 2 = 18 combinations here.
xgb_param_grid = {
    "n_estimators"  : [100, 200, 300],      # number of trees
    "max_depth"     : [4, 6, 8],            # tree depth — controls overfitting
    "learning_rate" : [0.05, 0.10],         # step size — lower = more trees needed
    # "subsample"   : [0.8, 1.0],           # uncomment to add more params
    # "colsample_bytree": [0.8, 1.0],
}

# ── StratifiedKFold preserves fraud % in every fold ──────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator  = XGBClassifier(
        random_state      = 42,
        use_label_encoder = False,
        eval_metric       = "logloss",
        n_jobs            = -1,
    ),
    param_grid = xgb_param_grid,
    cv         = cv,
    scoring    = "roc_auc",     # metric used to compare combinations
    n_jobs     = -1,
    verbose    = 1,             # prints progress; set to 0 to silence
    refit      = True,          # auto-refits best model on full trainval data
)

grid_search.fit(X_trainval_clean, y_trainval)

# ── Read the output — this answers all your tuning questions ──────────────────
print(f"\n── GridSearchCV results ──")
print(f"Best parameters : {grid_search.best_params_}")
print(f"Best val AUC    : {grid_search.best_score_:.4f}")

# Full results table — sort by mean_test_score to see all combinations ranked
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df[[
    "param_n_estimators", "param_max_depth", "param_learning_rate",
    "mean_test_score", "std_test_score", "rank_test_score"
]].sort_values("rank_test_score")

print("\n── All combinations tried ──")
print(results_df.to_string(index=False))

# ── Visual: val AUC by max_depth ──────────────────────────────────────────────
pivot = results_df.pivot_table(
    index="param_max_depth",
    columns="param_n_estimators",
    values="mean_test_score"
)
pivot.plot(kind="bar", figsize=(10, 5), ylim=(0.88, 0.95))
plt.title("Validation AUC by max_depth and n_estimators")
plt.xlabel("max_depth")
plt.ylabel("Mean CV AUC")
plt.legend(title="n_estimators")
plt.tight_layout()
plt.show()

# ── Overfitting check: train AUC vs val AUC for each depth ───────────────────
# A large gap signals the model memorised training data.
print("\n── Overfitting check (train AUC vs best val AUC per depth) ──")
for depth in xgb_param_grid["max_depth"]:
    temp_model = XGBClassifier(
        max_depth         = depth,
        n_estimators      = grid_search.best_params_["n_estimators"],
        learning_rate     = grid_search.best_params_["learning_rate"],
        use_label_encoder = False,
        eval_metric       = "logloss",
        random_state      = 42,
    )
    temp_model.fit(X_train_clean, y_train)
    train_auc = roc_auc_score(y_train, temp_model.predict_proba(X_train_clean)[:, 1])
    val_auc   = roc_auc_score(y_val,   temp_model.predict_proba(X_val_clean)[:, 1])
    gap       = train_auc - val_auc
    flag      = " ← OVERFIT" if gap > 0.05 else ""
    print(f"  max_depth={depth}  train={train_auc:.4f}  val={val_auc:.4f}  gap={gap:.4f}{flag}")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 — FINAL MODEL: REFIT ON ALL LABELLED DATA
# ─────────────────────────────────────────────────────────────────────────────
# Now that tuning is complete, val data is no longer needed for decisions.
# We refit on train + val combined to give the model as much data as possible
# before the final test evaluation.
#
# grid_search.best_estimator_ is already refitted (because refit=True above),
# but it was fitted on X_trainval_clean which is what we want. Use it directly.

final_model = grid_search.best_estimator_
print(f"\nFinal model parameters: {grid_search.best_params_}")

# Save the final model
joblib.dump(final_model, "fraud_final_model.pkl")
print("Final model saved → fraud_final_model.pkl")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7 — FINAL EVALUATION ON TEST SET (done exactly once)
# ─────────────────────────────────────────────────────────────────────────────
# X_test_clean was transformed using training pipeline parameters.
# We never used test data to make any decision above.
# This score is your honest, final performance number.

y_test_prob = final_model.predict_proba(X_test_clean)[:, 1]
y_test_pred = final_model.predict(X_test_clean)

test_auc = roc_auc_score(y_test, y_test_prob)
print(f"\n── FINAL TEST RESULTS ──")
print(f"Test AUC : {test_auc:.4f}")

# Compare to best CV val AUC — they should be close.
# A big drop signals the val evaluation was somehow optimistic.
print(f"CV val AUC was : {grid_search.best_score_:.4f}")
print(f"Gap            : {grid_search.best_score_ - test_auc:.4f}")

print("\n── Classification Report ──")
print(classification_report(y_test, y_test_pred, target_names=["Legit", "Fraud"]))

cm   = confusion_matrix(y_test, y_test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Legit", "Fraud"])
disp.plot(cmap="Blues")
plt.title("Final model — test set confusion matrix")
plt.show()


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8 — EXPLAINABILITY
# ─────────────────────────────────────────────────────────────────────────────

feat_importances = pd.Series(final_model.feature_importances_, index=X_train_clean.columns)
feat_importances.nlargest(15).sort_values().plot(kind="barh", figsize=(10, 6))
plt.title("Top 15 features — final XGBoost model")
plt.tight_layout()
plt.show()

explainer   = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_val_clean)   # use val for SHAP — test stays blind
shap.summary_plot(shap_values, X_val_clean)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9 — PRODUCTION INFERENCE
# ─────────────────────────────────────────────────────────────────────────────
# In a new notebook or production service, all you need are these two files.
# The pipeline handles all cleaning. The model handles all predictions.
# No manual preprocessing. No re-fitting anything.

# pipeline     = joblib.load("fraud_preprocessing_pipeline.pkl")
# final_model  = joblib.load("fraud_final_model.pkl")
#
# raw_new_data  = pd.read_csv("new_transactions.csv")
# X_new_clean   = pipeline.transform(raw_new_data)
# X_new_clean   = X_new_clean.drop(columns=["date"], errors="ignore")
# predictions   = final_model.predict(X_new_clean)
# probabilities = final_model.predict_proba(X_new_clean)[:, 1]